# Semantic Search Engine Interactive Activity
*AIPI 510: Data Sourcing for Analytics*

*Week 13*



---



**Build Your Own Semantic Search Engine**

> **Goal**: Create a semantic search app from
scratch and deploy it to Hugging Face Spaces.


1.   Pick a dataset
2.  Chunk + embed your data
3. Store your data in a Chroma vector db
4. Build a search app using Streamlit or Gradio
5. Publish on Hugging Face Spaces
6. Submit Link on Canvas



---

*This is a starter notebook. This is designed to give you a jumping off point only. *

Make sure to
`pip install chromadb gradio sentence-transformers`


In [ ]:
# Load an embedding model (you can use any embedding model)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

     |████████████████████████████████| 760 kB 1.4 MB/s eta 0:00:01
  Using cached scikit_learn-1.7.2-cp310-cp310-macosx_12_0_arm64.whl (8.7 MB)
     |████████████████████████████████| 857 kB 20.7 MB/s eta 0:00:01
  Using cached scipy-1.15.3-cp310-cp310-macosx_14_0_arm64.whl (22.4 MB)
     |████████████████████████████████| 80 kB 6.2 MB/s eta 0:00:01
     |████████████████████████████████| 127.2 MB 205 kB/s  eta 0:00:01
  Using cached numpy-2.2.6-cp310-cp310-macosx_14_0_arm64.whl (5.3 MB)
     |████████████████████████████████| 12.8 MB 11.8 MB/s eta 0:00:01
     |████████████████████████████████| 3.1 MB 10.9 MB/s eta 0:00:01
     |████████████████████████████████| 96 kB 20.4 MB/s eta 0:00:01
     |████████████████████████████████| 221 kB 80.4 MB/s eta 0:00:01
     |████████████████████████████████| 133 kB 41.8 MB/s eta 0:00:01
     |████████████████████████████████| 126 kB 17.2 MB/s eta 0:00:01
  Using cached pyyaml-6.0.3-cp310-cp310-macosx_11_0_arm64.whl (174 kB)
  Using cached tomli-

/Users/justin/duke/aipi/510_data_sourcing/semantic-search-project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1108 22:43:20.745187 3750 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19698.75it/s]


In [ ]:
# Load the National Park Service dataset built by build_dataset.py
# (NPS Data API: park overviews, activities/topics, weather, things to do, campgrounds).
# Build with:  NPS_API_KEY=... python build_dataset.py   |   re-chunk only:  python build_dataset.py --chunk-only
import json

with open("data/nps_chunks.json") as f:
    data = json.load(f)

texts = data["texts"]          # each chunk starts with a header like "Yosemite National Park (CA) | Park overview"
metadatas = data["metadatas"]  # doc_type, park_name, states, designation, url, latitude/longitude, ... (for Chroma's metadatas=)
ids = data["ids"]              # stable ids derived from NPS record ids

print(data["info"])
print(texts[0], "\n", metadatas[0])

In [ ]:
# Generate embeddings
embeddings = model.encode(texts)

In [ ]:
# Store embeddings in ChromaDB
import chromadb
from chromadb.config import Settings
client = chromadb.Client(Settings(anonymized_telemetry=False))
collection = client.create_collection("my_collection")
collection.add(documents=texts, embeddings=embeddings.tolist(), metadatas=metadatas, ids=ids)

In [ ]:
# Define a search function
def semantic_search(query):
    query_embedding = model.encode([query])
    results = collection.query(query_embeddings=query_embedding.tolist(), n_results=1)
    return "\n\n".join(results['documents'][0])

In [ ]:
# Simple Gradio interface
import gradio as gr
demo = gr.Interface(
    fn=semantic_search,
    inputs=gr.Textbox(label="Enter your search query"),
    outputs=gr.Textbox(label="Top Matches"),
    title="Semantic Search Engine",
    description="Search over your custom dataset using semantic similarity."
)

demo.launch()

### Next Steps:

- Save your chromadb
- Write code to load your chromadb
- Convert your loading, search function and app code to an app.py script
- Create a requirements.txt file
- Test your code!!
- Push your git repo to Hugging Face Spaces


___
Your file tree in your repository will look like this:

- chromadb/
  - chroma.sqlite3
- README.md
- requirements.txt
- app.py